In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.0 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-pose.pt')

# Train on the augmented dataset
model.train(
    data='/content/drive/MyDrive/TubeDetectionDataset_ZeonSystems2026/pose_config.yaml',
    epochs=40,
    imgsz=640,
    batch=16,
    name='tube_pose_s_model',
    degrees=0.0,
    flipud=0.0,
    fliplr=0.0,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/TubeDetectionDataset_ZeonSystems2026/pose_config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, h

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ed155ddf050>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034, 

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# ==========================================
# 1. TEST CONFIGURATION (Update these paths)
# ==========================================
POSE_MODEL_PATH = "/content/best_s.pt"
TEST_IMG_DIR = "/content/drive/MyDrive/TubeDetectionDataset_ZeonSystems2026/augmented_images"
TEST_CSV_PATH = "/content/drive/MyDrive/TubeDetectionDataset_ZeonSystems2026/augmented_annotations.csv"

MATCH_THRESHOLD = 30.0 # Max pixel distance to consider a match valid

print("Loading model and test data...")
pose_model = YOLO(POSE_MODEL_PATH)
df_test = pd.read_csv(TEST_CSV_PATH)

# ==========================================
# 2. CORE LOGIC
# ==========================================
def get_angle_from_pose(kpts):
    jx, jy = kpts[0] # Joint
    tx, ty = kpts[2] # Tab
    dx = tx - jx
    dy = jy - ty
    angle_rad = math.atan2(dy, dx)
    return (math.degrees(angle_rad) + 360) % 360

def run_hybrid_inference(img_path):
    img = cv2.imread(img_path)
    results = pose_model(img, verbose=False)[0]
    predictions = []

    if results.keypoints is None or results.keypoints.xy is None or results.boxes is None:
        return predictions

    kpts_array = results.keypoints.xy.cpu().numpy()
    boxes_array = results.boxes.xyxy.cpu().numpy()

    for kpts, box in zip(kpts_array, boxes_array):
        if len(kpts) < 3: continue

        # --- A. Get YOLO Coarse Prediction ---
        jx, jy = kpts[0]
        cx, cy = kpts[1]
        tx, ty = kpts[2]

        dx = tx - jx
        dy = jy - ty
        yolo_angle = (math.degrees(math.atan2(dy, dx)) + 360) % 360

        # --- B. Get Bounding Box & Crop ---
        x1, y1, x2, y2 = map(int, box)
        pad = 5
        x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
        x2, y2 = min(img.shape[1], x2 + pad), min(img.shape[0], y2 + pad)

        crop = img[y1:y2, x1:x2]
        final_angle = yolo_angle

        if crop.size > 0:
            # --- C. OpenCV Ellipse Fitting ---
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            corners_sum = int(thresh[0,0]) + int(thresh[0,-1]) + int(thresh[-1,0]) + int(thresh[-1,-1])
            if corners_sum > 510:
                thresh = cv2.bitwise_not(thresh)

            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                main_contour = max(contours, key=cv2.contourArea)
                if len(main_contour) >= 5:
                    ellipse = cv2.fitEllipse(main_contour)
                    e_angle = ellipse[2]

                    math_angle = (90 - e_angle) % 180
                    cand1 = math_angle
                    cand2 = (math_angle + 180) % 360

                    # --- D. Resolve 180 Ambiguity ---
                    diff1 = get_angle_error(cand1, yolo_angle)
                    diff2 = get_angle_error(cand2, yolo_angle)

                    final_angle = cand1 if diff1 < diff2 else cand2

        predictions.append([cx, cy, final_angle])

    return predictions

def get_angle_error(pred_deg, gt_deg):
    diff = abs(pred_deg - gt_deg)
    return min(diff, 360.0 - diff)

def evaluate_predictions(preds, gts):
    if len(preds) == 0:
        return 0, 0, len(gts), []

    preds_arr, gts_arr = np.array(preds), np.array(gts)
    dist_matrix = cdist(preds_arr[:, :2], gts_arr[:, :2])
    row_ind, col_ind = linear_sum_assignment(dist_matrix)

    tp, fp, fn = 0, 0, 0
    angle_errors = []
    matched_preds, matched_gts = set(), set()

    for r, c in zip(row_ind, col_ind):
        if dist_matrix[r, c] <= MATCH_THRESHOLD:
            tp += 1
            matched_preds.add(r)
            matched_gts.add(c)
            err = get_angle_error(preds_arr[r, 2], gts_arr[c, 2])
            angle_errors.append(err)

    fp = len(preds) - len(matched_preds)
    fn = len(gts) - len(matched_gts)
    return tp, fp, fn, angle_errors

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
unique_images = df_test['image'].unique()
print(f"Running Hybrid Inference on {len(unique_images)} images...\n")

total_tp, total_fp, total_fn = 0, 0, 0
all_angle_errors = []
missing_images = 0

for img_name in unique_images:
    gt_rows = df_test[df_test['image'] == img_name]
    gts = gt_rows[['center_x', 'center_y', 'angle_deg']].values.tolist()

    img_path = os.path.join(TEST_IMG_DIR, img_name)

    if not os.path.exists(img_path):
        missing_images += 1
        total_fn += len(gts)
        continue

    preds = run_hybrid_inference(img_path)

    tp, fp, fn, errs = evaluate_predictions(preds, gts)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    all_angle_errors.extend(errs)

if missing_images > 0:
    print(f"⚠️ WARNING: {missing_images} images from the CSV were NOT found in the folder!\n")

# ==========================================
# 4. FINAL REPORT
# ==========================================
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
mae = np.mean(all_angle_errors) if all_angle_errors else 0

print(f"--- HYBRID (YOLOv8s + Ellipse) RESULTS ---")
print(f"Centers Detected: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")
print(f"Angle Mean Abs Error (MAE): {mae:.2f} degrees")
print(f"(TP: {total_tp}, FP: {total_fp}, FN: {total_fn})")

Loading model and test data...
Running Hybrid Inference on 350 images...

--- HYBRID (YOLOv8s + Ellipse) RESULTS ---
Centers Detected: Precision=0.928, Recall=1.000, F1=0.963
Angle Mean Abs Error (MAE): 30.52 degrees
(TP: 1852, FP: 143, FN: 0)
